# Read Me
This notebook will create a structure under your selected data catalog and schema. Please ensure to configure as required.
There are a few options here for:
- motor insurance
- property insurance
- commercial insurance

Each of the tables will have a relevant prefix.
Global tables are relevant for every business line.


This notebook is structured in the  following way:
- global configuration
- global_ tables shared between product lines
  - like customers or postcodes
- motor_ tables
- property_ tables
- commercial_ tables

Each business line has a simple rating engine. The quotes are first generated and then priced using that rating engine (new column is added).


# Global Configuration

In [0]:
# Configuration
catalog_name = "lrcatalog"
schema_name = "agentic_underwriting_dev"
user_path = "laurence.ryszka@databricks.com"

# Create catalog and schema
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")


In [0]:
%pip install faker

# Global Tables
shared between product lines

### Create Postcodes table

In [0]:
# # Postcodes table
# # Adjust this to match the actual path in your repo
# file_path = "Repo/Users/laurence.ryszka@databricks.com/actuarial-pricing-demo/Agentic-Motor-underwriting/data/ONSPD_MAY_2025_UK_CR.csv"  # relative path from notebook location

# # Load CSV using Spark
# df = spark.read.option("header", True).csv(file_path)

# # Save to Unity Catalog
# table_name = "uk_postcodes"
# df.write.mode("overwrite").saveAsTable(f"{catalog_name}.{schema_name}.{table_name}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog_name}.{schema_name}.global_uk_postcodes AS
SELECT * FROM lrcatalog.agentic_underwriting.uk_postcodes
""")

### Drop All Tables (for dev)

In [0]:
# # Assumes catalog_name and schema_name already defined
# catalog = catalog_name
# schema  = schema_name

# # List all tables in the schema
# tables = [t.name for t in spark.catalog.listTables(f"{catalog}.{schema}")]
# #display(tables)
# # Drop each table
# for t in tables:
#     fqtn = f"{catalog}.{schema}.{t}"
#     spark.sql(f"DROP TABLE IF EXISTS {fqtn}")
#     print(f"Dropped: {fqtn}")

### Create Global Customer Table

In [0]:
# GLOBAL: Customers master shared across LoBs -> agentic_underwriting.global_customers
import random
import pandas as pd
from faker import Faker

faker = Faker("en_GB")
random.seed(42)

# Assumes you already have: catalog_name, schema_name
num_customers = 15116  # keep aligned with uk_postcodes for deterministic demos

# Use real postcodes from your table for geographic realism
pc_df = spark.table(f"{catalog_name}.{schema_name}.global_uk_postcodes").select("pcd")
postcode_list = [r.pcd for r in pc_df.limit(num_customers).collect()]

customers = []
for i, pcd in enumerate(postcode_list):
    first, last = faker.first_name(), faker.last_name()
    line1 = faker.street_address()
    # Optional line2 about ~30% of the time
    line2 = faker.secondary_address() if random.random() < 0.3 else None
    city = faker.city()
    county = faker.county()
    email = f"{first}.{last}{i}@example.com".lower().replace(" ", "")
    phone = faker.phone_number()
    age = random.randint(18, 90)

    customers.append({
        "customer_id": f"C{100000+i}",
        "first_name": first,
        "last_name": last,
        "age": age,                      # NEW field
        "address_line1": line1,
        "address_line2": line2,
        "city": city,
        "county": county,
        "postcode": pcd,                 # from uk_postcodes for consistency
        "email": email,
        "phone": phone
    })

customers_df = pd.DataFrame(customers)
customers_sdf = spark.createDataFrame(customers_df)
customers_sdf.write.mode("overwrite").saveAsTable(f"{catalog_name}.{schema_name}.global_customers")

display(spark.table(f"{catalog_name}.{schema_name}.global_customers").limit(10))

### Create table for agent run outputs

In [0]:
#Create empty table for agent output
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog_name}.{schema_name}.global_agent_output (
  quote_id STRING,
  agent_output STRING)
""")

### Create table for call transcripts 

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

table_fqtn = f"{catalog_name}.{schema_name}.global_sales_call_transcripts"

# Define schema
schema = StructType([
    StructField("quote_id", StringType(), False),
    StructField("call_transcript", StringType(), True),
])

# Create empty DF and save as a managed table (create if not exists)
empty_df = spark.createDataFrame([], schema)
if not spark.catalog.tableExists(table_fqtn):
    empty_df.write.mode("overwrite").saveAsTable(table_fqtn)
else:
    print(f"Table already exists: {table_fqtn}")

### Create table for confirming NCD and claims amount
This is Lexis Nexis API simulator

In [0]:
#global_claims_disclosure_validation

import pandas as pd
import random

def tbl(name: str) -> str:
    return f"{catalog_name}.{schema_name}.{name}"

def table_exists(fqtn: str) -> bool:
    return spark.catalog.tableExists(fqtn)

# Build GLOBAL claims_disclosure_validation from available quotes
motor_rows = []
property_rows = []

# Load motor quotes if present
if table_exists(tbl("motor_quotes")):
    mqt = spark.table(tbl("motor_quotes")).selectExpr(
        "quote_id",
        "'motor' as lob",
        "first_name",
        "last_name",
        "postcode",
        "CAST(ncd_amount AS INT) as ncd_amount",
        "CAST(claims_amount AS INT) as declared_claims"
    )
    motor_rows = mqt.toPandas().to_dict("records")

# Load property quotes if present
if table_exists(tbl("property_quotes")):
    pqt = spark.table(tbl("property_quotes")).selectExpr(
        "quote_id",
        "'property' as lob",
        "first_name",
        "last_name",
        "postcode",
        "CAST(0 AS INT) as ncd_amount",                    # (demo) NCD not used for property
        "CAST(previous_claims AS INT) as declared_claims"  # map to generic field
    )
    property_rows = pqt.toPandas().to_dict("records")

combined_rows = motor_rows + property_rows

# Fallback: if no quotes yet, seed a tiny placeholder set
if not combined_rows:
    # use a few postcodes from the global set if available
    if table_exists(tbl("global_uk_postcodes")):
        pc_list = [r.pcd for r in spark.table(tbl("global_uk_postcodes")).select("pcd").limit(10).collect()]
    else:
        pc_list = [f"PC{i:03d} 1AA" for i in range(10)]

    combined_rows = [{
        "quote_id": f"PLH{i+1:04d}",
        "lob": random.choice(["motor", "property"]),
        "first_name": "Demo",
        "last_name": "User",
        "postcode": pc,
        "ncd_amount": 0,
        "declared_claims": random.choice([0, 1])
    } for i, pc in enumerate(pc_list)]

# Add a simple validation effect: actual_claims = declared +/- 1, floored at 0
for r in combined_rows:
    r["actual_claims"] = max(0, r["declared_claims"] + random.choice([-1, 0, 1]))

global_claims_df = pd.DataFrame(combined_rows)
spark.createDataFrame(global_claims_df).write.mode("overwrite").saveAsTable(tbl("global_claims_disclosure_validation"))

# Motor insurance data

## Generate quotes

In [0]:
import random
import pandas as pd


num_quotes = 15116  # must match number of postcodes in global_uk_postcodes

# Get list of real postcodes
postcode_df = spark.table(f"{catalog_name}.{schema_name}.global_uk_postcodes").select("pcd")
postcode_list = [row.pcd for row in postcode_df.limit(num_quotes).collect()]

# Get customers (assume global_customers has: customer_id, first_name, last_name, age)
customers_df = spark.table(f"{catalog_name}.{schema_name}.global_customers") \
    .select("customer_id", "first_name", "last_name", "age") \
    .limit(num_quotes)
customers_list = customers_df.collect()

# Reference lists
vehicle_types = ["Hatchback", "SUV", "Sedan", "Hot Hatch", "Van"]
storage_options = ["garage", "driveway", "street"]
channels = ["aggregator", "direct"]

# Generate synthetic quote rows by zipping postcodes with customers
quotes_data = []
for i in range(len(postcode_list)):
    customer = customers_list[i]
    vehicle = random.choice(vehicle_types)
    storage = random.choice(storage_options)
    ncd = random.choice([0, 1, 3, 5, 10])
    claims = random.choice([0, 1, 2])
    pcd = postcode_list[i]

    quotes_data.append({
        "quote_id": f"Q{1000+i}",
        "customer_id": customer.customer_id,
        "first_name": customer.first_name,
        "last_name": customer.last_name,
        "age": customer.age,
        "postcode": pcd,
        "vehicle_type": vehicle,
        "storage_declared": storage,
        "ncd_amount": ncd,
        "claims_amount": claims,
        "channel": random.choice(channels)
    })

# Convert and save to Unity Catalog
quotes_df = pd.DataFrame(quotes_data)
quotes_sdf = spark.createDataFrame(quotes_df)
quotes_sdf.write.mode("overwrite").saveAsTable(f"{catalog_name}.{schema_name}.motor_quotes")

## Enrichment tables

In [0]:
# MOTOR enrichment tables
# Assumes:
# catalog_name = "lrcatalog"
# schema_name = "agentic_underwriting_dev"

import pandas as pd
import random

def tbl(name: str) -> str:
    return f"{catalog_name}.{schema_name}.{name}"

# motor_driver_risk_profile (age-based bands)
ages = list(range(18, 101))
risk_segments = []
for age in ages:
    if age <= 21:
        risk_segments.append("very_high_risk")
    elif age <= 30:
        risk_segments.append("high_risk")
    elif age <= 60:
        risk_segments.append("medium_risk")
    elif age <= 75:
        risk_segments.append("high_risk")
    else:
        risk_segments.append("very_high_risk")

motor_driver_df = pd.DataFrame({"age": ages, "risk_segment": risk_segments})
spark.createDataFrame(motor_driver_df).write.mode("overwrite").saveAsTable(tbl("motor_driver_risk_profile"))

# motor_vehicle_risk_attributes (static)
vehicle_types = ["Hatchback", "SUV", "Sedan", "Hot Hatch", "Van"]
vehicle_risk_data = []
for v in set(vehicle_types):
    risk_band = random.choice(["low", "medium", "high"])
    is_high_value = (v in ["Hot Hatch", "SUV"]) and (risk_band == "high")
    vehicle_risk_data.append({
        "vehicle_type": v,
        "vehicle_risk_band": risk_band,
        "is_high_value": is_high_value
    })

motor_vehicle_df = pd.DataFrame(vehicle_risk_data)
spark.createDataFrame(motor_vehicle_df).write.mode("overwrite").saveAsTable(tbl("motor_vehicle_risk_attributes"))

## Add call transcripts

In [0]:
from pyspark.sql import Row

table_fqtn = f"{catalog_name}.{schema_name}.global_sales_call_transcripts"

t1 = """Agent: Good morning, you're speaking with Amy from Swift Insurance. How can I help you today?

Customer: Hi, I'd like to get a quote for my car insurance.

Agent: Sure. Can I take your name, please?

Customer: John Doe.

Agent: Thanks, Mr. Doe. Can I confirm your postcode?

Customer: CR3 6JF.

Agent: And the type of vehicle?

Customer: It's an SUV.

Agent: Great. Where is the vehicle usually stored?

Customer: On the driveway.

Agent: Got it. And how many years of no claims discount do you have?

Customer: 6 years.

Agent: And how many claims have you had in the last 5 years?

Customer: Three.

Agent: Thank you. Just a moment while I generate your quote.

...

Agent: The quote I have for you today is £1332.

Customer: Okay, thanks for your help."""
r1 = Row(quote_id="MR9999", call_transcript=t1)

t2 = """Agent: Good afternoon, you're speaking with Olivia from Swift Insurance. How can I help today?

Customer: Hi, I'm looking to get a quote for my car insurance.

Agent: Of course. Can I take your name, please?

Customer: Lauren Fish.

Agent: Thank you, Ms. Fish. What's your postcode?

Customer: CR3 6JF.

Agent: Got it. What type of vehicle is it?

Customer: It's a Hatchback.

Agent: And where is the vehicle usually kept?

Customer: On the driveway.

Agent: Perfect. How many years of no claims discount do you have?

Customer: 5 years.

Agent: And any claims in the past 5 years?

Customer: None.

Agent: Thanks. Let me just generate your quote...

...

Agent: Alright, the quote I have for you is £450.

Customer: That sounds good. Thank you!"""
r2 = Row(quote_id="MR9998", call_transcript=t2)

spark.createDataFrame([r1, r2]).write.mode("append").saveAsTable(table_fqtn)

## Rating Engine

In [0]:
from pyspark.sql.functions import col, when, round as round_col

quotes_table = f"{catalog_name}.{schema_name}.motor_quotes"

# Load table
quotes_df = spark.table(quotes_table)

# Define multipliers as column expression
vehicle_multiplier = when(col("vehicle_type") == "SUV", 1.2) \
    .when(col("vehicle_type") == "Hatchback", 1.0) \
    .when(col("vehicle_type") == "Sports", 1.5) \
    .otherwise(1.1)

# Define age adjustment
age_adjustment = when(col("age") < 25, 200) \
    .when(col("age") < 35, 100) \
    .otherwise(0)

# Define storage adjustment
storage_adjustment = when(col("storage_declared") == "garage", -50) \
    .when(col("storage_declared") == "driveway", 0) \
    .when(col("storage_declared") == "street", 50) \
    .otherwise(25)

# Final price formula with all adjustments and multiplier
base_price_expr = (
    500 +
    (col("claims_amount") * 100) +
    age_adjustment +
    storage_adjustment -
    (col("ncd_amount") * 30)
)

# Apply vehicle multiplier and round
priced_df = quotes_df.withColumn(
    "quote_value",
    round_col(base_price_expr * vehicle_multiplier, 2)
)

# Overwrite the table with updated prices
priced_df.write.option("mergeSchema", "true").mode("overwrite").saveAsTable(quotes_table)

## Quotes for the demo
Add a quote we will base our demo on, must be added to quotes and claims disclosure validation table.
(need to rewrite the story)

In [0]:
from pyspark.sql import Row

# --- Config ---
catalog = catalog_name
schema  = schema_name

# --- 1) Append motor quotes (MR9998, MR9997, MR9999) ---
motor_quotes_rows = [
    Row(
        quote_id="MR9998", first_name="Lauren", last_name="Fish", age=61,
        postcode="CR3 6JF", vehicle_type="Hatchback", storage_declared="driveway",
        ncd_amount=5, claims_amount=0, channel="direct", quote_value=float(450)
    ),
    Row(
        quote_id="MR9997", first_name="James", last_name="Bond", age=61,
        postcode="CR3 6JE", vehicle_type="Hot Hatch", storage_declared="garage",
        ncd_amount=2, claims_amount=0, channel="direct", quote_value=float(450)
    ),
    Row(
        quote_id="MR9999", first_name="John", last_name="Doe", age=32,
        postcode="CR3 6JF", vehicle_type="SUV", storage_declared="driveway",
        ncd_amount=3, claims_amount=6, channel="direct", quote_value=float(1332)
    )
]

spark.createDataFrame(motor_quotes_rows).write.mode("append").saveAsTable(
    f"{catalog}.{schema}.motor_quotes"
)

# --- 2) Append to global claims_disclosure_validation (3 rows) ---
global_claims_rows = [
    Row(first_name="Lauren", last_name="Fish", postcode="CR3 6JF", ncd_amount=5, claims_amount=0),
    Row(first_name="James",  last_name="Bond", postcode="CR3 6JE", ncd_amount=2, claims_amount=0),
    Row(first_name="John",   last_name="Doe",  postcode="CR3 6JF", ncd_amount=6, claims_amount=3),
]

spark.createDataFrame(global_claims_rows).write \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable(f"{catalog}.{schema}.global_claims_disclosure_validation")

# --- 3) Upsert property_attributes for two postcodes via replaceWhere ---
property_updates = [
    Row(postcode="CR3 6JF", has_garage=False, has_driveway=True,  property_risk_level="mid_high"),
    Row(postcode="CR3 6JE", has_garage=False, has_driveway=False, property_risk_level="mid_high"),
]

spark.createDataFrame(property_updates).write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("replaceWhere", "postcode IN ('CR3 6JF','CR3 6JE')") \
    .saveAsTable(f"{catalog}.{schema}.property_attributes")

# Property insurance data

## Generate quotes

In [0]:
import random
import pandas as pd

# Variables already set above
# catalog_name = "lrcatalog"
# schema_name = "agentic_underwriting_dev"

num_quotes = 15116  # must match number of postcodes in global_uk_postcodes

# Get list of real postcodes
postcode_df = spark.table(f"{catalog_name}.{schema_name}.global_uk_postcodes").select("pcd")
postcode_list = [row.pcd for row in postcode_df.limit(num_quotes).collect()]

# Get customers (assume global_customers has: customer_id, first_name, last_name, age)
customers_df = spark.table(f"{catalog_name}.{schema_name}.global_customers") \
    .select("customer_id", "first_name", "last_name", "age") \
    .limit(num_quotes)
customers_list = customers_df.collect()

# Reference lists for property
property_types = ["Detached", "Semi-Detached", "Terraced", "Flat", "Bungalow"]
construction_types = ["Brick", "Timber", "Concrete", "Stone"]
roof_types = ["Tile", "Slate", "Thatched", "Flat Roof"]
occupancy_status = ["Owner-Occupied", "Tenant", "Vacant"]
channels = ["aggregator", "direct"]

# Generate synthetic property quote rows
quotes_data = []
for i in range(len(postcode_list)):
    customer = customers_list[i]
    pcd = postcode_list[i]

    quotes_data.append({
        "quote_id": f"PQ{1000+i}",
        "customer_id": customer.customer_id,
        "first_name": customer.first_name,
        "last_name": customer.last_name,
        "age": customer.age,
        "postcode": pcd,
        "property_type": random.choice(property_types),
        "construction_type": random.choice(construction_types),
        "roof_type": random.choice(roof_types),
        "num_bedrooms": random.randint(1, 6),
        "year_built": random.randint(1900, 2022),
        "occupancy_status": random.choice(occupancy_status),
        "previous_claims": random.choice([0, 1, 2, 3]),
        "channel": random.choice(channels)
    })

# Convert and save to Unity Catalog
quotes_df = pd.DataFrame(quotes_data)
quotes_sdf = spark.createDataFrame(quotes_df)

quotes_sdf.write \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.{schema_name}.property_quotes")


## Enrichment tables

In [0]:
# PROPERTY enrichment tables
# Assumes:
# catalog_name = "lrcatalog"
# schema_name = "agentic_underwriting_dev"

import pandas as pd
import random

def tbl(name: str) -> str:
    return f"{catalog_name}.{schema_name}.{name}"

# Read postcodes from global
postcode_df = spark.table(tbl("global_uk_postcodes")).select("pcd")
postcode_list = [row.pcd for row in postcode_df.collect()]
num_rows = len(postcode_list)

# Simple, ordered risk bands across the postcode list (demo)
risk_levels = ["high", "mid_high", "mid_low", "low"]
num_levels = len(risk_levels)

risk_per_block = num_rows // num_levels
remaining = num_rows % num_levels

ordered_risks = []
for i in range(num_levels):
    count = risk_per_block + (1 if i < remaining else 0)
    ordered_risks.extend([risk_levels[i]] * count)

property_data = []
for i, pc in enumerate(postcode_list):
    property_data.append({
        "postcode": pc,
        "has_garage": random.choice([True, False]),
        "has_driveway": random.choice([True, False]),
        "property_risk_level": ordered_risks[i]
    })

property_attributes_df = pd.DataFrame(property_data)
spark.createDataFrame(property_attributes_df).write.mode("overwrite").saveAsTable(tbl("global_property_attributes"))

## Add call transcripts

In [0]:
from pyspark.sql import Row

catalog = catalog_name
schema  = schema_name

# ===============================
# 1) PROPERTY call transcripts
# ===============================
table_transcripts = f"{catalog}.{schema}.global_sales_call_transcripts"

t_pr9999 = """Agent: Good morning, this is Daniel from Swift Insurance. How can I help?

Customer: Hi, I’d like a quote for my home insurance.

Agent: Certainly. Your name and postcode?

Customer: Sarah Miller, CR2 7AB.

Agent: Thanks. Property details?

Customer: Detached house, built 1995, four bedrooms. I’ve had 2 claims in the last five years. The house is detached 4 bedroom property. 

Agent: (typing) Okay, noted… What are other details about the property?

Customer: It's built of brick, owner-occupied and has tiled roof. Built in 1995. 

Agent: Thanks a lot for all the details - your quote for this is house is £XXX.

Customer: Thank you, good bye!

"""

t_pr9998 = """Agent: Hello, this is James from Swift Insurance. How can I assist today?

Customer: Hi, I need a quote for my house insurance.

Agent: Of course. Your name and postcode?

Customer: Tom Evans, CR5 4XZ.

Agent: Thanks, Mr. Evans. I’ll note: property is a bungalow, built around 2008, two bedrooms. It has a tiled roof. Is it owner-occupied and have you had any claims in the last 5 years?

Customer: Owner-occupied, and no claims.

Agent: Perfect. Let me calculate that now…

...

Agent: Your premium is £XXX.

Customer: That works for me, thanks!"""


spark.createDataFrame([
    Row(quote_id="PR9999", call_transcript=t_pr9999),
    Row(quote_id="PR9998", call_transcript=t_pr9998),
]).write.mode("append").saveAsTable(table_transcripts)


## Quotes for the demo
Add a quote we will base our demo on, must be added to quotes and claims disclosure validation table. (need to rewrite the story)

In [0]:
# ---------------------------
# 2) PROPERTY quotes (match the transcripts)
# ---------------------------
table_quotes = f"{catalog}.{schema}.property_quotes"

property_quotes_rows = [
    # PR9999 — roof corrected to Flat Roof (more expensive), previous_claims = 1 in last 5y
    Row(
        quote_id="PR9999",
        first_name="Sarah",
        last_name="Miller",
        age=45,
        postcode="CR2 7AB",
        property_type="Detached",
        construction_type="Brick",
        roof_type="Tile",          # corrected based on agent enrichment
        num_bedrooms=2,
        year_built=1995,
        occupancy_status="Owner-Occupied",
        previous_claims=4,
        channel="direct"
    ),
    # PR9998 
    Row(
        quote_id="PR9998",
        first_name="Tom",
        last_name="Evans",
        age=38,
        postcode="CR5 4XZ",
        property_type="Bungalow",
        construction_type="Concrete",
        roof_type="Tile",          
        num_bedrooms=2,
        year_built=2008,
        occupancy_status="Owner-Occupied",
        previous_claims=0,
        channel="direct"
      
    )
]

spark.createDataFrame(property_quotes_rows).write.mode("append").saveAsTable(table_quotes)

## Rating Engine

In [0]:
from pyspark.sql.functions import col, when, round as round_col, coalesce, lit
from pyspark.sql import functions as F

# Tables
quotes_table = f"{catalog_name}.{schema_name}.property_quotes"
attrs_table  = f"{catalog_name}.{schema_name}.property_attributes"

# Load quotes
quotes_df = spark.table(quotes_table)

# Optionally join postcode-level attributes (risk band, garage/driveway)
if spark.catalog.tableExists(attrs_table):
    attrs_df = spark.table(attrs_table).select(
        "postcode", "property_risk_level", "has_garage", "has_driveway"
    )
    quotes_df = quotes_df.join(attrs_df, on="postcode", how="left")
else:
    quotes_df = quotes_df.withColumn("property_risk_level", lit(None).cast("string")) \
                         .withColumn("has_garage", lit(False)) \
                         .withColumn("has_driveway", lit(False))

# --- Multipliers & adjustments ---

# Property type multiplier
prop_type_mult = (
    when(col("property_type") == "Detached", 1.10)
    .when(col("property_type") == "Semi-Detached", 1.05)
    .when(col("property_type") == "Terraced", 1.00)
    .when(col("property_type") == "Flat", 0.95)
    .when(col("property_type") == "Bungalow", 1.00)
    .otherwise(1.00)
)

# Risk band multiplier (from property_attributes)
risk_mult = (
    when(col("property_risk_level") == "high", 1.30)
    .when(col("property_risk_level") == "mid_high", 1.15)
    .when(col("property_risk_level") == "mid_low", 1.05)
    .when(col("property_risk_level") == "low", 1.00)
    .otherwise(1.00)
)

# Construction adjustment
construction_adj = (
    when(col("construction_type") == "Timber", 100)
    .when(col("construction_type") == "Concrete", 50)
    .when(col("construction_type") == "Stone", 20)
    .when(col("construction_type") == "Brick", 0)
    .otherwise(25)
)

# Roof adjustment
roof_adj = (
    when(col("roof_type") == "Thatched", 300)
    .when(col("roof_type") == "Flat Roof", 100)
    .when(col("roof_type") == "Slate", 20)
    .when(col("roof_type") == "Tile", 0)
    .otherwise(25)
)

# Occupancy adjustment
occupancy_adj = (
    when(col("occupancy_status") == "Vacant", 200)
    .when(col("occupancy_status") == "Tenant", 100)
    .when(col("occupancy_status") == "Owner-Occupied", 0)
    .otherwise(50)
)

# Bedrooms adjustment (centered around 3 bedrooms, +/- £20 per room)
bedrooms_adj = (coalesce(col("num_bedrooms"), lit(3)) - lit(3)) * lit(20)

# Property age adjustment
year_built = coalesce(col("year_built"), lit(1995))
age_adj = (
    when(year_built < 1950, 150)
    .when(year_built < 1975, 100)
    .when(year_built < 2000, 50)
    .otherwise(0)
)

# Previous claims adjustment
claims_adj = coalesce(col("previous_claims"), lit(0)) * lit(150)

# Protection discounts from attributes
garage_disc   = when(col("has_garage") == True, -30).otherwise(0)
driveway_disc = when(col("has_driveway") == True, -10).otherwise(0)

# Optional channel tweak (tiny uplift for aggregator)
channel_adj = when(col("channel") == "aggregator", 15).otherwise(0)

# Base price and final
base_price_expr = (
    lit(300) + construction_adj + roof_adj + occupancy_adj +
    bedrooms_adj + age_adj + claims_adj + garage_disc + driveway_disc + channel_adj
)

priced_df = quotes_df.withColumn(
    "quote_value",
    round_col(base_price_expr * prop_type_mult * risk_mult, 2)
)

# Write back
priced_df.write.option("mergeSchema", "true").mode("overwrite").saveAsTable(quotes_table)

# Commercial insurance

## Generate quotes

In [0]:
import random
import pandas as pd

# Variables already set above
# catalog_name = "lrcatalog"
# schema_name = "agentic_underwriting_dev"

num_quotes = 15116  # must match number of postcodes in global_uk_postcodes

# Get list of real postcodes
postcode_df = spark.table(f"{catalog_name}.{schema_name}.global_uk_postcodes").select("pcd")
postcode_list = [row.pcd for row in postcode_df.limit(num_quotes).collect()]

# Get customers (assume global_customers has: customer_id, first_name, last_name, age)
customers_df = spark.table(f"{catalog_name}.{schema_name}.global_customers") \
    .select("customer_id", "first_name", "last_name", "age") \
    .limit(num_quotes)
customers_list = customers_df.collect()

# Reference lists for commercial insurance
industries = ["Retail", "Hospitality", "Construction", "Manufacturing", "IT Services", "Logistics"]
lines_of_business = ["Property", "Liability", "Combined", "Cyber"]
channels = ["broker", "direct"]
deductibles = [250, 500, 1000, 2500, 5000]
limits = [100000, 250000, 500000, 1000000, 5000000]

# Generate synthetic commercial quote rows
quotes_data = []
for i in range(len(postcode_list)):
    customer = customers_list[i]
    pcd = postcode_list[i]

    quotes_data.append({
        "quote_id": f"CQ{1000+i}",
        "customer_id": customer.customer_id,
        "company_name": f"{customer.last_name} & Co",
        "industry": random.choice(industries),
        "line_of_business": random.choice(lines_of_business),
        "postcode": pcd,
        "turnover": random.randint(100_000, 10_000_000),     # annual turnover £
        "num_employees": random.randint(1, 500),
        "sum_insured": random.choice(limits),
        "deductible": random.choice(deductibles),
        "previous_claims": random.choice([0, 1, 2, 3]),
        "channel": random.choice(channels)
    })

# Convert and save to Unity Catalog
quotes_df = pd.DataFrame(quotes_data)
quotes_sdf = spark.createDataFrame(quotes_df)

quotes_sdf.write \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.{schema_name}.commercial_quotes")

## Enrichment Tables

In [0]:
# COMMERCIAL enrichment tables
# Assumes:
# catalog_name = "lrcatalog"
# schema_name = "agentic_underwriting_dev"

import pandas as pd
import random

def tbl(name: str) -> str:
    return f"{catalog_name}.{schema_name}.{name}"

# ---------------------------------------------------------
# 1) Industry × Turnover band risk profile
# ---------------------------------------------------------
industries = ["Retail", "Hospitality", "Construction", "Manufacturing", "IT Services", "Logistics"]
turnover_bands = [
    ("micro",       0,        250_000),
    ("small",       250_000,  2_000_000),
    ("medium",      2_000_000,10_000_000),
    ("large",       10_000_000, 50_000_000),
    ("enterprise",  50_000_000, 1_000_000_000)
]

# simple mapping baseline by industry
industry_base = {
    "Retail":        ("medium_risk", 1.05),
    "Hospitality":   ("high_risk",   1.15),
    "Construction":  ("high_risk",   1.20),
    "Manufacturing": ("mid_high",    1.10),
    "IT Services":   ("mid_low",     0.98),
    "Logistics":     ("high_risk",   1.18),
}

rows = []
for ind in industries:
    base_seg, base_mult = industry_base[ind]
    for band_name, lo, hi in turnover_bands:
        # nudge multiplier by band (bigger firms get slightly lower volatility, but bigger limits)
        band_adj = {
            "micro": 1.08, "small": 1.04, "medium": 1.00, "large": 0.98, "enterprise": 0.97
        }[band_name]
        rows.append({
            "industry": ind,
            "turnover_band": band_name,
            "turnover_min": lo,
            "turnover_max": hi,
            "risk_segment": base_seg,
            "base_rate_multiplier": round(base_mult * band_adj, 3)
        })

df_industry_risk = pd.DataFrame(rows)
spark.createDataFrame(df_industry_risk) \
    .write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(tbl("commercial_industry_risk_profile"))

# ---------------------------------------------------------
# 2) Postcode-level protection & territory risk
# ---------------------------------------------------------
# Read postcodes from global
postcode_df = spark.table(tbl("global_uk_postcodes")).select("pcd")
postcode_list = [r.pcd for r in postcode_df.collect()]
num_rows = len(postcode_list)

# Simple, ordered risk bands across the postcode list (demo)
risk_levels = ["high", "mid_high", "mid_low", "low"]
num_levels = len(risk_levels)
risk_per_block = num_rows // num_levels
remaining = num_rows % num_levels

ordered_risks = []
for i in range(num_levels):
    count = risk_per_block + (1 if i < remaining else 0)
    ordered_risks.extend([risk_levels[i]] * count)

prot_rows = []
alarm_grades = ["None", "Bells Only", "Monitored", "Police Response"]
for i, pc in enumerate(postcode_list):
    prot_rows.append({
        "postcode": pc,
        "has_sprinklers": random.choice([True, False, False]),  # skew to False
        "alarm_grade": random.choice(alarm_grades),
        "fire_resistance_class": random.choice(["Low", "Medium", "High"]),
        "territory_risk_level": ordered_risks[i]
    })

df_protection = pd.DataFrame(prot_rows)
spark.createDataFrame(df_protection) \
    .write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(tbl("commercial_property_protection"))

# ---------------------------------------------------------
# 3) Liability exposure bands (employees → rate)
# ---------------------------------------------------------
emp_bands = [
    (0, 5,    "low",       45.0),
    (6, 20,   "mid_low",   42.0),
    (21, 50,  "mid_high",  40.0),
    (51, 200, "high",      38.0),
    (201, 999999, "very_high", 36.0)
]
liab_rows = []
for lo, hi, seg, rate in emp_bands:
    liab_rows.append({
        "employees_min": lo,
        "employees_max": hi,
        "liability_risk_segment": seg,
        "rate_per_employee": rate  # demo £ per employee basis
    })

df_liability = pd.DataFrame(liab_rows)
spark.createDataFrame(df_liability) \
    .write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(tbl("commercial_liability_exposure_bands"))

## Call Transcripts

In [0]:
from pyspark.sql import Row

table_fqtn = f"{catalog_name}.{schema_name}.global_sales_call_transcripts"

# --- Transcript 1: CR9999 ---
t1 = """Agent: Good morning, this is Alice from Swift Insurance. How can I help you today?

Customer: Hi, I’m looking to get a quote for my company’s insurance.

Agent: Of course. May I take your company name?

Customer: Miller Catering Ltd.

Agent: Thank you. What’s the postcode of your main premises?

Customer: CR2 7AB.

Agent: Great. What line of business are you in?

Customer: Hospitality, we run a chain of restaurants.

Agent: Understood. How many employees do you have?

Customer: About 45.

Agent: And your annual turnover?

Customer: Around £2 million.

Agent: Have you had any claims in the past five years?

Customer: One small fire claim, about £25,000.

Agent: Thank you. One moment while I generate your quote...

...

Agent: The premium I have for you today is £4,850.

Customer: Alright, thank you for your help."""
r1 = Row(quote_id="CR9999", call_transcript=t1)

# --- Transcript 2: CR9998 ---
t2 = """Agent: Good afternoon, this is Daniel from Swift Insurance. How can I help today?

Customer: Hello, I need a quote for liability insurance for my construction business.

Agent: Certainly. May I take your company name?

Customer: Evans Builders Ltd.

Agent: Thanks. And the postcode?

Customer: CR5 4XZ.

Agent: Got it. Approximately how many employees do you have?

Customer: 120.

Agent: And your annual turnover?

Customer: Around £8 million.

Agent: Do you have any US exposure?

Customer: No, we only operate in the UK.

Agent: Any claims in the past five years?

Customer: None.

Agent: Great. Let me calculate your premium...

...

Agent: The premium I can offer you today is £12,300.

Customer: Sounds good, thank you."""
r2 = Row(quote_id="CR9998", call_transcript=t2)

# Append both rows
spark.createDataFrame([r1, r2]).write.mode("append").saveAsTable(table_fqtn)

## Quotes for the demo


In [0]:
from pyspark.sql import Row

# --- Config ---
catalog = catalog_name
schema  = schema_name
table_fqtn = f"{catalog}.{schema}.commercial_quotes"

# --- Quotes for CR9999 (Miller Catering Ltd) and CR9998 (Evans Builders Ltd) ---
quotes_rows = [
    Row(
        quote_id="CR9999",
        customer_id="CUST_MILLER",           # demo ID, could map to global_customers
        company_name="Miller Catering Ltd",
        industry="Hospitality",
        line_of_business="Combined",
        postcode="CR2 7AB",
        turnover=2_000_000,
        num_employees=45,
        sum_insured=1_000_000,
        deductible=1000,
        previous_claims=1,
        channel="broker" 
    ),
    Row(
        quote_id="CR9998",
        customer_id="CUST_EVANS",
        company_name="Evans Builders Ltd",
        industry="Construction",
        line_of_business="Liability",
        postcode="CR5 4XZ",
        turnover=8_000_000,
        num_employees=120,
        sum_insured=5_000_000,
        deductible=2500,
        previous_claims=0,
        channel="broker"
        
    )
]

# Append to commercial_quotes
spark.createDataFrame(quotes_rows).write.mode("append").saveAsTable(table_fqtn)

## Rating Engine

In [0]:
from pyspark.sql.functions import col, when, round as round_col, lit, coalesce

# Tables
quotes_table = f"{catalog_name}.{schema_name}.commercial_quotes"

# Load existing quotes
quotes_df = spark.table(quotes_table)

# --- Multipliers & adjustments ---

# Industry multiplier
industry_mult = (
    when(col("industry") == "Construction", 1.25)
    .when(col("industry") == "Hospitality", 1.15)
    .when(col("industry") == "Retail", 1.10)
    .when(col("industry") == "Logistics", 1.20)
    .when(col("industry") == "Manufacturing", 1.12)
    .when(col("industry") == "IT Services", 0.95)
    .otherwise(1.05)
)

# Size adjustment (employees)
size_adj = (
    when(col("num_employees") < 10, 200)
    .when(col("num_employees") < 50, 500)
    .when(col("num_employees") < 200, 1000)
    .otherwise(2000)
)

# Turnover adjustment
turnover_adj = (
    when(col("turnover") < 500_000, 250)
    .when(col("turnover") < 2_000_000, 500)
    .when(col("turnover") < 10_000_000, 1000)
    .otherwise(2000)
)

# Claims adjustment
claims_adj = coalesce(col("previous_claims"), lit(0)) * 500

# Deductible adjustment (higher deductible reduces premium)
deductible_adj = - (col("deductible") / 50)

# Sum insured adjustment (scale premium by exposure)
sum_insured_factor = (col("sum_insured") / 1_000_000) * 200

# Channel adjustment
channel_adj = when(col("channel") == "broker", 100).otherwise(0)

# --- Base premium expression ---
base_price_expr = (
    lit(1000) + size_adj + turnover_adj + claims_adj + deductible_adj +
    sum_insured_factor + channel_adj
)

# Apply multipliers and round
priced_df = quotes_df.withColumn(
    "quote_value",
    round_col(base_price_expr * industry_mult, 2)
)

# Overwrite table with new column added
priced_df.write.option("mergeSchema", "true").mode("overwrite").saveAsTable(quotes_table)